# Workflow for paper 1

Uses paper-1-version branch of zoonosim

We start by importing zoonosim and some other libraries we may need

In [9]:
import zoonosim as zn
import numpy as np
import seaborn as sns
import pandas as pd
import optuna as op
import matplotlib.pyplot as plt
from scipy import stats
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from matplotlib.legend_handler import HandlerTuple

Next we set up the options and project name

In [2]:
project_name = "Paper_1"
zn.options.set(verbose=2) # How much output to print to the console (0 = none, 1 = some, 2 = all)
zn.options.set(numba_parallel='safe') # Whether to use parallel processing for numba-compiled functions. Options are 'safe', 'force', or 'off'.
zn.options.set(numba_cache=False) # Whether to cache numba-compiled functions to disk. Options are True or False.

Reloading Zoonosim so changes take effect...
Zoonosim 0.0.2 (2026-03-03) — © 2025 by McGill University
Reload complete. Note: for some options to take effect, you may also need to delete Zoonosim's __pycache__ folder.
Reloading Zoonosim so changes take effect...
Zoonosim 0.0.2 (2026-03-03) — © 2025 by McGill University
Reload complete. Note: for some options to take effect, you may also need to delete Zoonosim's __pycache__ folder.


Now we define our baseline parameters

In [3]:

baseline_pars = dict(
    # Peliminary parameters
    rand_seed = 73, # Random seed for the simulation. This is used to ensure that the results are reproducible.
    pop_scale = 1.0, # Scale factor for the population size. This is a vestigial parameter that is no longer used, but is kept for compatibility with older code.
    rescale = False, # Whether to rescale the population size to match the pop_size parameter. This is a vestigial parameter that is no longer used, but is kept for compatibility with older code.
    record_all_events = False, # Whether to record all events in the simulation. We set it to False during calibration to save memory.
    verbose = zn.options.get('verbose'), # How much output to print to the console (0 = none, 1 = some, 2 = all)

    # Simulation parameters

    agent_types = ['human', 'ppe', 'flock', 'barn', 'water'],
    n_farms = 50,
    pop_size = None, # This gets filled in when the population is generated
    pop_size_by_type = {}, # This gets filled in when the population is generated
    pop_pars = dict( # Parameters used for generating the population
        avg_barns_per_farm = 5.0, # Average number of barns per farm. Used to define the number of barns in the population. The actual number of barns per farm is drawn from a Poisson distribution with this mean.
        avg_humans_per_barn = 1.5, # We define the number of humans per barn rather than per farm to insue larger farms have more humans. The actual number of humans per barn is drawn from a Poisson distribution with this mean.
        avg_water_per_farm = 0.75, # Average number of water sources per farm. The actual number of water sources per farm is drawn from a Poisson distribution with this mean. This value should be less than 1.0 to ensure that some farms share water sources.
        number_of_transients = 3, # Number of transient agents in the population. These are agents that move between farms and can spread disease between them.
        visits_per_day = 3, # Number of farms each transient visits in a day
    ),

    initial_conditions = dict( # dictates how many of each agent type are initially infected at the start of the simulation
        human = 0,
        ppe = 0,
        flock = 0,
        barn = 0,
        water = 0
    ),
    start_day = '2022-01-01', # The earliest day that we have data to calibrate the model to. This is used to set the start day of the simulation.
    end_day = '2025-12-31', # The latest day that we have data to calibrate the model to. This is used to set the end day of the simulation.
    n_days = None, # The number of days to run the simulation. If None, it will be set to the number of days between start_day and end_day.
    n_beds_hosp = None, # The number of hospital beds available in the population. If None, there is no restriction.
    no_hosp_factor = 1.0, # Change in mortality rate if hospital beds are not available. This is a multiplicative factor applied to the baseline mortality rate.
    beta = dict( # Beta values for each agent type.
        human = 0.0,
        ppe = 0.0,
        flock = 0.0,
        barn = 0.0,
        water = 0.0,
    ),
    n_imports = dict(
        human=None,  # Number of imported human cases per day; None = disabled
        flock=None,  # Number of imported flock cases per day; None = disabled
        ppe = None,   # Number of imported PPE contaminations per day; None = disabled
        barn=dict(import_pattern='seasonal', max_import_rate=0.2, peak_day=300),  # Number of imported barn contaminations per day; None = disabled
        water=dict(import_pattern='seasonal', max_import_rate=0.2, peak_day=300),  # Number of imported water contaminations per day; None = disabled
    ),
    transmission_pars = dict(
        human = dict(
                beta_dist = dict(dist='neg_binomial', par1=1.0, par2=0.45, step=0.01), # Distribution to draw individual level transmissibility
                viral_loads = dict(minimum_detectable_load=3, peak_load=6),
                viral_levels = dict(min_scl=0.25, max_scl=1.0), # Specifies the range within which viral load should be scaled so it can contribute to relative transmissibility
                gamma_pars = dict(shape=2.0, scale=0.35) # Parameters for the gamma distribution used to determine the moment when an agents viral load exceeds the minimum detectable load. This is used to determine when an agent becomes infectious.
        ),
        ppe = dict(
            beta_dist = dict(dist='neg_binomial', par1=1.0, par2=0.45, step=0.01)
        ),
        flock = dict(
            beta_dist = dict(dist='neg_binomial', par1=1.0, par2=0.45, step=0.01)
        ),
        barn = dict(
            beta_dist = dict(dist='neg_binomial', par1=1.0, par2=0.45, step=0.01)
        ),
        water = dict(
            beta_dist = dict(dist='neg_binomial', par1=1.0, par2=0.45, step=0.01)
        )
    ),
    immunity_pars = dict(
        human = dict(
        use_waning = True, # Whether or not to use waning immunity; set to True for humans by default
        nab_init = dict(dist='normal', par1=0, par2=2),  # Parameters for the distribution of the initial level of log2(nab) following natural infection, taken from fig1b of https://doi.org/10.1101/2021.03.09.21252641
        nab_decay = dict(form='nab_growth_decay', growth_time=21, decay_rate1=np.log(2) / 50, decay_time1=150, decay_rate2=np.log(2) / 250, decay_time2=365), # NOTE: I have no idea where this comes from
        nab_kin = None, # Constructed during sim initialization using the nab_decay parameters
        nab_boost = 1.5, # Multiplicative factor applied to a person's nab levels if they get reinfected.
        nab_eff = dict(alpha_inf=1.08, alpha_inf_diff=1.812, beta_inf=0.967, alpha_symp_inf=-0.739, beta_symp_inf=0.038, alpha_sev_symp=-0.014, beta_sev_symp=0.079), # Parameters to map nabs to efficacy
        rel_imm_symp = dict(asymp=0.85, mild=1, severe=1.5), # Relative immunity from natural infection varies by symptoms.
        immunity = None, # Matrix of immunity and cross-immunity factors, set by init_immunity() in immunity.py
        trans_redux = 0.59 # Reduction in transmission for breakthrough infection
        ),
        ppe = dict(use_waning = False),
        flock = dict(use_waning = False),
        barn = dict(use_waning = False),   
        water = dict(use_waning = False)
    ),
    dur = dict(
        human = dict(        # Duration: disease progression
            exp2inf = dict(dist='lognormal_int', par1=3.0, par2=1.5), # Duration from exposed to infectious
            inf2sym = dict(dist='lognormal_int', par1=1.5, par2=0.5), # Duration from infectious to symptomatic
            sym2sev = dict(dist='lognormal_int', par1=5.0, par2=2.0), # Duration from symptomatic to severe symptoms

            # Duration: Recovery
            asym2rec = dict(dist='lognormal_int', par1=8.0,  par2=2.0), # Duration for asymptomatic people to recover
            mild2rec = dict(dist='lognormal_int', par1=8.0,  par2=2.0), # Duration for people with mild symptoms to recover
            sev2rec = dict(dist='lognormal_int', par1=14.0, par2=6.0), # Duration for people with severe symptoms to recover
            sev2die = dict(dist='lognormal_int', par1=10.0, par2=5.0), # Duration from critical symptoms to death, 18.8 days total

            # Duration: quarantine
            quar = 7,
            # Duration: diagnosis
            diag = 14
        ),
            ppe = dict(        
            contamination = dict(dist='lognormal_int', par1=14, par2=5.0), # Duration of contamination.
            quar = 7, # Duration of quarantine after suspected exposure. NOTE: This should generally be the same as the human quarantine duration.
        ),
        flock = dict(
            # Duration: disease progression
            exp2inf = dict(dist='lognormal_int', par1=1.5, par2=0.5), # Duration from exposed to infectious. 
            inf2out = dict(dist='lognormal_int', par1=2.0, par2=1.0), # Duration from infectious to recovery/removal.
            susp2res = dict(dist='lognormal_int', par1=5.0, par2=1.0), # Duration from suspicion to a definitive test result. 

            # Duration: Quarantine
            quar = 14
        ),
        barn = dict(
            contamination = dict(dist='lognormal_int', par1=14, par2=5.0), # Duration of contamination. 
            composting = dict(dist='lognormal_int', par1=7.0, par2=1.5), # Duration of composting. 
            cleaning = dict(dist='lognormal_int', par1=7.0, par2=1.5), # Duration of cleaning process. 
        ),
        water = dict(
             contamination = dict(dist='lognormal_int', par1=14, par2=5.0), # Duration of contamination. 
        )
    ),
    poultry_pars = dict(
        breeds = np.array(['poultry'], dtype=zn.default_str),
        breed_freqs = np.array([1.0]),
        mortality_suspicion_threshold = [0.0012], # I.E a deviation from the expected mortality rate of 0.01*expected_value will trigger suspicion
        symptomatic_suspicion_threshold = [0.0001], # I.E a deviation from the expected symptomatic rate of 0.01*expected_value will trigger suspicion
        consumption_suspicion_threshold = [0.001], # I.E a deviation from the expected rate of water consumption of 0.01*expected_value will trigger suspicion
        cycle_dur = [dict(dist = 'normal_pos', par1 = 100, par2 = 25)],
        flock_size = [dict(dist = 'normal_pos', par1 = 20000, par2 = 10000)]
    ),
    prognoses = dict(
        human = dict(
            age_cutoffs   = np.array([0]),     # Age cutoffs (lower limits)
            sus_ORs       = np.array([1.00]),    # Odds ratios for relative susceptibility 
            trans_ORs     = np.array([0.00]),    # Odds ratios for relative transmissibility
            comorbidities = np.array([1.00]),    # Comorbidities by age -- set to 1 by default since already included in disease progression rates
            symp_probs    = np.array([0.66]),    # relative probability of developing symptoms 
            severe_probs  = np.array([0.33]),     # relative probability of developing severe symptoms
            death_probs   = np.array([0.33]),    # relative probability of dying
        ),
        ppe = dict(
            sus_ORs = np.array([1.00]),
            trans_ORs = np.array([1.00]),
        ),
        flock = dict(
            breeds = np.array(['poultry'], dtype=zn.default_str),
            sus_ORs = np.array([1.00]),
            trans_ORs = np.array([1.00]),
            baseline_symptomatic_rate = np.array([0.001]),
            symptomatic_rate_increase = np.array([dict(dist='lognormal', par1 = 0.001, par2 = 0.5)]),
            baseline_mortality_rate = np.array([0.001]),
            mortality_rate_increase = np.array([dict(dist='lognormal', par1 = 0.005, par2 = 0.5)]),
            baseline_water_rate = np.array([1.00]),
            water_rate_increase = np.array([dict(dist='lognormal', par1 = 1.5, par2 = 0.5)]),
        ),
        barn = dict(
            sus_ORs = np.array([1.00]),
            trans_ORs = np.array([1.00]),
        ),
        water = dict(
            sus_ORs = np.array([0.00]),
            trans_ORs = np.array([1.00]),
        )
    ),

    # Smartwatch parameters
    enable_smartwatches = False,
    smartwatch_pars = dict(
        who                       = 'all', # Must be one of 'all', 'permanent', 'transient'; controls who receives smartwatches
        mean_fpr                  =   0.08, # mean false positive rate
        use_variable_fpr          = True, # Whether to use a variable false positive rate
        day_i                     = np.arange(-21, 22, 1), #
        loc                       = 3.25, # Day of max probability of alert, relative to the day of symptom onset.
        alpha                     = 1, # Scales the probability of receiving an alert
        usage_rate                = 1, # Out of people who have smartwatches, the amount who use download the alerting app and stick with it.
        compliance_rate           =   0.05, # probability of quarantining if a smartwatch detects symptoms (only used if testobjs are not available)
        participation_rate        =   0.3,  # proportion of the population that has a smartwatch
    ),

    bkg_ILI = 0.0, # proportion of the population that is infected with Influenza-like-illness at any given time
    Avian_to_ILI = False, # If True agents can be infected with both Avian Influenza and an ILI

    # Bells and whistles

    interventions = [], # The interventions present in this simulation; populated by the user
    surveillance = [], # The surveillance systems present in this simulation; populated by the user
    testing = [], # The testing systems present in this simulation; populated by the user. These can be populated externally, or internally by supplying a parameter dictionary. 
    analyzers = [], # Custom analysis functions; populated by the user
    timelimit = None, # Time limit for the simulation (seconds)
    stopping_func = None, # A function to call to stop the sim partway through
    vaccine_pars = {}, # Vaccines that are being used; populated during initialization
    vaccine_map = {}, # Reverse mapping from number to vaccine key

    # Pathogen specific parameters
    wild = dict(
        human = dict(
            rel_beta = 1.0,
            rel_symp_prob = 0.33,
            rel_severe_prob = 0.25,
            rel_death_prob = 0.01,
            rel_asymp_fact = 0.5

        ),
        ppe = dict(
            rel_beta = 1.0,
            rel_dur_contamination = 1.0
        ),
        flock = dict(
            rel_beta = 1.0,
            rel_symp_delta = 1.0,
            rel_death_delta = 1.0,
            rel_water_delta = 1.0
        ),
        barn = dict(
            rel_beta = 1.0,
            rel_dur_contamination = 1.0
        ),
        water = dict(
            rel_beta = 1.0,
            rel_dur_contamination = 1.0
        )
    ),

    variants = [], # Additional variants of the virus; populated by the user, see immunity.py
    variant_map = {0:'wild'}, # Reverse mapping from number to variant key
    variant_pars = dict(
        wild = dict(
            human = dict(
                rel_beta = 1.0,
                rel_symp_prob = 0.33,
                rel_severe_prob = 0.25,
                rel_death_prob = 0.01,
                rel_asymp_fact = 0.5

            ),
            ppe = dict(
                rel_beta = 1.0,
                rel_dur_contamination = 1.0
            ),
            flock = dict(
                rel_beta = 1.0,
                rel_symp_delta = 1.0,
                rel_death_delta = 1.0,
                rel_water_delta = 1.0
            ),
            barn = dict(
                rel_beta = 1.0,
                rel_dur_contamination = 1.0
            ),
            water = dict(
                rel_beta = 1.0,
                rel_dur_contamination = 1.0
            )
        ),
    ),
)

zn.reset_layer_pars(baseline_pars)

## Calibration Step

Now we need to prepare the data that we will be calibrating against.

In [4]:
original_data = pd.read_csv("../zoonosim/data/CFIA_original_dataset.csv")
clean_data = original_data.rename(columns={'Infected Premises (IP) location Question': 'Infected premises', 
                                              'Premises type Question': 'Premises type',
                                              'WOAH premises classification Question': 'Premises classification',
                                              'Primary control zone (PCZ) Question - Map': 'PCZ',
                                              'Status of order declaring PCZ': 'PCZ status'})
clean_data['Infected premises'] = clean_data['Infected premises'].str.extract(r"^([A-Z]{2}-IP\d+)")
clean_data = clean_data[clean_data['Premises type'] == 'commercial']
clean_data = clean_data[clean_data['Premises classification'] == 'poultry']

manipulated_data = clean_data[['Date detected', 'Province']]
manipulated_data = manipulated_data.rename(columns={'Date detected': 'date', 'Province': 'province'})
manipulated_data['date'] = pd.to_datetime(manipulated_data['date'], format='%B %d, %Y')

daily_timeseries = (
    manipulated_data
    .sort_values("date")
    .groupby(["date", "province"])
    .size()
    .unstack(fill_value=0)              # daily counts per province   
    .asfreq("D", fill_value=0)          # ensure every date appears
    #.reset_index('date', drop=False)
)

full_date_range = pd.date_range(start=daily_timeseries.index.min().replace(day = 1, month=1), end=daily_timeseries.index.max(), freq='D')

daily_timeseries = daily_timeseries.reindex(full_date_range, fill_value=0).reset_index().rename(columns={'index': 'date'})

daily_timeseries['date'] = pd.to_datetime(daily_timeseries['date'])
monthly_timeseries = daily_timeseries.groupby(pd.Grouper(key='date', freq='ME')).sum().reset_index()

no_poultry_farms_QC = 1000 # TODO: get the actual number from lit rev
no_poultry_farms_CA = 10000 # TODO: get the actual number from lit rev
no_simulated_farms = baseline_pars['n_farms']

monthly_timeseries['Quebec_adjusted'] = (monthly_timeseries['Quebec']/no_poultry_farms_QC)*no_simulated_farms
monthly_timeseries['Canada'] = monthly_timeseries['British Columbia'] 
+ monthly_timeseries['Alberta'] 
+ monthly_timeseries['Saskatchewan'] 
+ monthly_timeseries['Manitoba'] 
+ monthly_timeseries['Ontario']
+ monthly_timeseries['Quebec']
+ monthly_timeseries['Nova Scotia']
monthly_timeseries['Canada_adjusted'] = (monthly_timeseries['Canada']/no_poultry_farms_CA)*no_simulated_farms


calibration_scope = 'Quebec' # We can fit either to Quebec only data of canada wide data 
calibration_column = calibration_scope + "_adjusted"
calibration_data = monthly_timeseries[['date', calibration_column]].copy()
calibration_data = calibration_data.rename(columns={calibration_column:"monthly_new_poultry_flock_infectious"})
calibration_data['monthly_new_human_infectious'] = np.repeat(0, len(calibration_data))



Now we create the simulation object

In [5]:
calibration_sim = zn.Sim(datafile = calibration_data, label = project_name, pars = baseline_pars)

If we want we can save these pars now

In [6]:
calibration_sim.export_pars(f"saved_pars/{project_name}.json")

{'agent_types': ['human', 'ppe', 'flock', 'barn', 'water'],
 'n_farms': 50,
 'pop_size': None,
 'pop_size_by_type': {},
 'pop_pars': {'avg_barns_per_farm': 5.0,
  'avg_humans_per_barn': 1.5,
  'avg_water_per_farm': 0.75,
  'number_of_transients': 3,
  'visits_per_day': 3},
 'pop_scale': 1.0,
 'rescale': False,
 'record_all_events': False,
 'initial_conditions': {'human': 0,
  'ppe': 0,
  'flock': 0,
  'barn': 0,
  'water': 0},
 'start_day': '2022-01-01',
 'end_day': '2025-12-31',
 'n_days': None,
 'rand_seed': 73,
 'verbose': 2,
 'enable_testobjs': False,
 'testobjs': None,
 'enable_smartwatches': False,
 'smartwatch_pars': {'who': 'all',
  'mean_fpr': 0.08,
  'use_variable_fpr': True,
  'day_i': array([-21, -20, -19, -18, -17, -16, -15, -14, -13, -12, -11, -10,  -9,
          -8,  -7,  -6,  -5,  -4,  -3,  -2,  -1,   0,   1,   2,   3,   4,
           5,   6,   7,   8,   9,  10,  11,  12,  13,  14,  15,  16,  17,
          18,  19,  20,  21]),
  'loc': 3.25,
  'alpha': 1,
  'usage_rate'

And the last step before calibration is to specify what parameters can be adjusted and in what range. For any given parameter that we want to calibrate against we need to provide a list of three values, the first value in the list is the 'expected', or 'starting' value, this is the value that Optuna will start it's calibration from. the second and third value are the minimum and maximum respectively, they define the range of values that optuna will explore during calibration. This can all get a little tricky when parameters are in nested dicts or are stratified, I've done my best to make it all automatic but there are still issues now and then. If things aren't working it probably means you will have to make adjustments in analysis.py and/or utils/pars_ops.py.

In [7]:
calibration_pars = dict(
    beta = dict(
        human = [0.01, 0.00, 0.25],
        ppe = [0.05, 0.00, 0.25],
        flock = [0.10, 0.00, 0.25],
        barn = [0.15, 0.00, 0.25],
        water = [0.20, 0.00, 0.25]
    ),
    n_imports = dict(
        barn = dict(
            max_import_rate = [0.2, 0.0, 0.5],
            peak_day = [275, 180, 365]
            ),
        water = dict(
            max_import_rate = [0.2, 0.0, 0.5],
            peak_day = [275, 180, 365]
        )
    ),
    beta_layer = dict(
        human_ppe = [0.02, 0.00, 0.50],
        human_human = [0.02, 0.00, 0.50],
        human_flock = [0.02, 0.00, 0.50],
        human_barn = [0.02, 0.00, 0.50],
        human_water = [0.02, 0.00, 0.50],
        ppe_ppe = [0.02, 0.00, 0.50],
        ppe_flock = [0.02, 0.00, 0.50],
        ppe_barn = [0.02, 0.00, 0.50],
        ppe_water = [0.02, 0.00, 0.50],
        flock_barn = [0.02, 0.00, 0.50],
        flock_water = [0.02, 0.00, 0.50],
        barn_water = [0.02, 0.00, 0.50],
        transient = [0.02, 0.00, 0.50]
    )

) 

Now we can create and execute the calibration object

In [ ]:
calibrator = zn.Calibration(calibration_sim, calibration_pars, name = project_name, n_reps = 10, total_trials=100, die=True, keep_db=True)

if __name__ == "__main__": # This is needed because Windows is weird. 
    calibrator.calibrate()

[I 2026-09-08 14:13:54,705] A new study created in Journal with name: Paper_1


Initializing sim with 50 farms for 1460 days


——————————————————————————————————————————————————————————————
  Running "Before calibration": 2022-01-01 ( 0/1460) (0.06 s) 
——————————————————————————————————————————————————————————————



——————————————————————————————————————————————————————————————
  Running "Before calibration": 2022-01-02 ( 1/1460) (0.08 s) 
——————————————————————————————————————————————————————————————



——————————————————————————————————————————————————————————————
  Running "Before calibration": 2022-01-03 ( 2/1460) (0.09 s) 
——————————————————————————————————————————————————————————————



——————————————————————————————————————————————————————————————
  Running "Before calibration": 2022-01-04 ( 3/1460) (0.10 s) 
——————————————————————————————————————————————————————————————



——————————————————————————————————————————————————————————————
  Running "Before calibration": 2022-01-05 ( 4/1460) (0.11 s) 
———————————————————————————————————————————————————————————

The database created by the calibration gets saved in the studies folder. If we want we can load it from there, for example to avoid redoing the calibration process

In [11]:
db_name = f'../studies/{project_name}.db'
study = op.load_study(study_name=project_name, storage=op.storages.JournalStorage(op.storages.journal.JournalFileBackend(db_name, op.storages.journal.JournalFileOpenLock(db_name))))

And now we are able to see the results of the calibration

In [15]:
all_trials = study.trials_dataframe()
all_trials['value'].describe()

count       96.000000
mean      1702.100160
std       5503.360663
min          4.000000
25%         57.250000
50%        137.807692
75%        262.730769
max      28270.000000
Name: value, dtype: float64

In [21]:
top_trials = all_trials[all_trials['value']<17.0]
len(top_trials)

5

In [22]:
top_trials

,number,value,datetime_start,datetime_complete,duration,params_beta.barn,params_beta.flock,params_beta.human,params_beta.ppe,params_beta.water,...,params_beta_layer.ppe_barn,params_beta_layer.ppe_flock,params_beta_layer.ppe_ppe,params_beta_layer.ppe_water,params_beta_layer.transient,params_n_imports.barn.max_import_rate,params_n_imports.barn.peak_day,params_n_imports.water.max_import_rate,params_n_imports.water.peak_day,state
25,25,15.307692,2026-09-08 14:18:31.280208,2026-09-08 14:22:34.122751,0 days 00:04:02.842543,0.241673,0.068253,0.095478,0.113214,0.248728,...,0.462694,0.118731,0.001079,0.174999,0.328983,0.013554,343.168718,0.005862,353.320228,COMPLETE
32,32,4.000000,2026-09-08 14:22:34.144073,2026-09-08 14:26:49.054714,0 days 00:04:14.910641,0.238717,0.073157,0.128732,0.106591,0.246975,...,0.498171,0.118565,0.011914,0.122156,0.231100,0.343274,345.215314,0.001286,361.062637,COMPLETE
40,40,4.000000,2026-09-08 14:23:24.088897,2026-09-08 14:27:37.334341,0 days 00:04:13.245444,0.072636,0.062948,0.115238,0.111283,0.240718,...,0.473164,0.131060,0.127019,0.416640,0.225104,0.363729,340.714400,0.000008,327.790874,COMPLETE
87,87,11.076923,2026-09-08 14:36:15.954089,2026-09-08 14:40:21.475554,0 days 00:04:05.521465,0.110060,0.061969,0.174206,0.000366,0.230721,...,0.047339,0.103242,0.059362,0.455624,0.241788,0.154266,296.198119,0.032390,302.726430,COMPLETE
88,88,10.153846,2026-09-08 14:36:18.857786,2026-09-08 14:40:20.006035,0 days 00:04:01.148249,0.100761,0.125635,0.177027,0.005318,0.228829,...,0.053414,0.093134,0.058008,0.465870,0.244711,0.121121,251.310819,0.032754,325.524220,COMPLETE


Now we have found our top performing trials we need to convert them into the correct format for parameters. This can be done manually but it's slow and tedious so we will endeavour to automate it here. Note that the automation function is designed for the specific parameters being calibrated here, if you change the calibration parameters this may not work.

In [27]:
top_pars = []

for index, row in top_trials.iterrows():
    trial_pars = dict(
        beta = dict(
            human = row['params_beta.human'],
            ppe = row['params_beta.ppe'],
            flock = row['params_beta.flock'],
            barn = row['params_beta.barn'],
            water = row['params_beta.water']
        ),
        n_imports = dict(
            barn = dict(
                max_import_rate = row['params_n_imports.barn.max_import_rate'],
                peak_day = row['params_n_imports.barn.peak_day']
            ),
            water = dict(
                max_import_rate = row['params_n_imports.water.max_import_rate'],
                peak_day = row['params_n_imports.water.peak_day']
            )
        ),
        beta_layer = dict(
        human_ppe = row['params_beta_layer.human_ppe'],
        human_human = row['params_beta_layer.human_human'],
        human_flock = row['params_beta_layer.human_flock'],
        human_barn = row['params_beta_layer.human_barn'],
        human_water = row['params_beta_layer.human_water'],
        ppe_ppe = row['params_beta_layer.ppe_ppe'],
        ppe_flock = row['params_beta_layer.ppe_flock'],
        ppe_barn = row['params_beta_layer.ppe_barn'],
        ppe_water = row['params_beta_layer.ppe_water'],
        flock_barn = row['params_beta_layer.flock_water'],
        flock_water = row['params_beta_layer.flock_water'],
        barn_water = row['params_beta_layer.barn_water'],
        transient = row['params_beta_layer.transient']
        )
    )
    top_pars.append(trial_pars)

## Baseline Multisim

Now that we have our calibrated parameters we can put them all in an msim object and run the baseline simulation.

In [31]:
ensemble_sims = []

for i in range(0,len(top_pars)):
    temp_label = f'{project_name}_sim_{i}'
    temp_sim = zn.Sim(datafile=calibration_data, label=temp_label, pars=baseline_pars)
    temp_sim.update_pars(pars = top_pars[i], recursive = True)
    ensemble_sims.append(temp_sim)

ensemble_msim = zn.MultiSim(ensemble_sims, n_runs = 100, verbose = zn.options.get('verbose'))

And now we can run the multisim.

In [ ]:
if __name__ == "__main__": # This is needed because Windows is weird. 
    ensemble_msim.run(keep_people = True, run_args = dict(auto_finalize = False))
    ensemble_msim.finalize()
    ensemble_msim.shrink()
    ensemble_msim.save(f'msims/{project_name}_w_data.msim')
    ensemble_msim.combine()
    ensemble_msim.plot() # We will do our own custom plots later, here I just want to verify that everything worked correctly.